# SDH exp_017 — adaptive-capacity dynamic specialists
exp14의 메인 모델과 LR80/LGBM20을 고정하고 specialist 용량 및 routing만 비교한다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'experiments').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'experiments').exists():
    raise RuntimeError('저장소 내부에서 실행해 주세요.')
EXP14_DIR = PROJECT_ROOT / 'experiments/SDH/exp_014_focal_lgbm_specialists'
EXP17_DIR = PROJECT_ROOT / 'experiments/SDH/exp_017_adaptive_specialists'
RESULTS_DIR = EXP17_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for path in (EXP14_DIR, EXP17_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import lgbm_experiment as exp14
import adaptive_specialist as adaptive
print('project root:', PROJECT_ROOT)

In [ ]:
train = pd.read_csv(PROJECT_ROOT / 'data/raw/train.csv')
genes = [column for column in train if column not in ('ID', 'SUBCLASS')]
labels = train['SUBCLASS'].to_numpy()
classes = np.asarray(sorted(np.unique(labels)))
SEEDS = (42, 52, 62)
MODEL_WEIGHT = 0.20
BASELINE_CASE = 'fixed_ll__hard'
MAIN_CASE = exp14.main_cases()['main_01_multiclass_balanced']
CASES = adaptive.case_catalog()
print('train:', train.shape, 'classes:', len(classes), 'cases:', len(CASES))
display(pd.DataFrame([vars(value) for value in adaptive.presets().values()]))
display(pd.DataFrame([vars(value) for value in CASES.values()]).head(20))

## 1. Seed 42 공통 fold 피처와 기준 모델

In [ ]:
prepared_42, labels_42, classes_42 = exp14.prepare_seed(train, genes, seed=42)
assert np.array_equal(labels_42, labels)
assert np.array_equal(classes_42, classes)
lr_42 = exp14.evaluate_lr_reference(prepared_42, labels, classes, seed=42)
main_42 = exp14.evaluate_main_case(prepared_42, labels, classes, MAIN_CASE, seed=42)
print('LR:', lr_42.summary['oof_f1_macro'])
print('main LGBM:', main_42.summary['oof_f1_macro'])

## 2. Seed 42 specialist bank 학습
각 fold의 두 pair에 small/medium/large를 한 번씩 학습한다. 이후 144개 case는 재학습하지 않는다.

In [ ]:
bank_42 = adaptive.fit_specialist_bank(prepared_42, seed=42)
bank_rows = []
for fold in bank_42:
    for rank, (pair, support) in enumerate(zip(fold.pairs, fold.pair_supports), start=1):
        bank_rows.append({'fold': fold.fold, 'rank': rank, 'pair': pair, 'support': support})
display(pd.DataFrame(bank_rows))

## 3. Seed 42 — 144 cases 전체 스크리닝

In [ ]:
leaderboard_42, results_42 = adaptive.evaluate_catalog(
    main_42, bank_42, labels, lr_42, model_weight=MODEL_WEIGHT,
)
leaderboard_42['delta_vs_exp14_seed42'] = (
    leaderboard_42['blend_f1']
    - float(leaderboard_42.loc[leaderboard_42['case'] == BASELINE_CASE, 'blend_f1'].iloc[0])
)
leaderboard_42.to_csv(RESULTS_DIR / 'seed42_144_case_screen.csv', index=False)
display(leaderboard_42.head(30))
display(leaderboard_42[leaderboard_42['case'] == BASELINE_CASE])

## 4. 3-seed 확인 후보 잠금
exp14 기준과 seed42 상위 8개를 확인한다. 순위가 같아도 case 이름으로 결정해 재현성을 유지한다.

In [ ]:
TOP_K = 8
ranked_cases = leaderboard_42.sort_values(
    ['blend_f1', 'specialist_f1', 'case'], ascending=[False, False, True]
)['case'].tolist()
CONFIRM_CASES = [BASELINE_CASE]
selected_probabilities = [results_42[BASELINE_CASE].probability]
for case_name in ranked_cases:
    if case_name not in CONFIRM_CASES:
        candidate_probability = results_42[case_name].probability
        if any(np.allclose(candidate_probability, chosen, atol=1e-12) for chosen in selected_probabilities):
            continue
        CONFIRM_CASES.append(case_name)
        selected_probabilities.append(candidate_probability)
    if len(CONFIRM_CASES) >= TOP_K + 1:
        break
print('confirmation cases:', CONFIRM_CASES)

## 5. Seed 52/62 확인
아래 셀은 두 seed의 fold 피처, LR, main LGBM과 specialist bank를 차례로 학습하므로 오래 걸린다.

In [ ]:
confirmation_rows = []
seed_objects = {42: {'lr': lr_42, 'main': main_42, 'bank': bank_42}}
for seed in SEEDS:
    print(f'\n===== confirmation seed {seed} =====')
    if seed == 42:
        lr_result, main_result, bank = lr_42, main_42, bank_42
    else:
        prepared, seed_labels, seed_classes = exp14.prepare_seed(train, genes, seed=seed)
        assert np.array_equal(seed_labels, labels)
        assert np.array_equal(seed_classes, classes)
        lr_result = exp14.evaluate_lr_reference(prepared, labels, classes, seed=seed)
        main_result = exp14.evaluate_main_case(prepared, labels, classes, MAIN_CASE, seed=seed)
        bank = adaptive.fit_specialist_bank(prepared, seed=seed)
        seed_objects[seed] = {'lr': lr_result, 'main': main_result, 'bank': bank}
    for case_name in CONFIRM_CASES:
        specialist = adaptive.apply_case(main_result, bank, labels, CASES[case_name])
        blend_probability = (1.0-MODEL_WEIGHT)*lr_result.probability + MODEL_WEIGHT*specialist.probability
        blend_prediction = classes[np.argmax(blend_probability, axis=1)]
        confirmation_rows.append({
            'seed': seed, 'case': case_name,
            'specialist_f1': specialist.summary['oof_f1_macro'],
            'blend_f1': f1_score(labels, blend_prediction, average='macro'),
            'routed_rows': int(specialist.fold_metrics['routed_rows'].sum()),
        })
    pd.DataFrame(confirmation_rows).to_csv(RESULTS_DIR / 'confirmation_partial.csv', index=False)
    print('checkpoint saved through seed', seed)
confirmation = pd.DataFrame(confirmation_rows)
confirmation.to_csv(RESULTS_DIR / 'confirmation_candidates_3seed.csv', index=False)
display(confirmation)

## 6. 최종 안정성 판정

In [ ]:
baseline_by_seed = confirmation[confirmation['case'] == BASELINE_CASE].set_index('seed')['blend_f1']
confirmation['delta_vs_exp14'] = confirmation.apply(
    lambda row: row['blend_f1'] - baseline_by_seed.loc[row['seed']], axis=1,
)
summary = confirmation.groupby('case', as_index=False).agg(
    blend_mean=('blend_f1', 'mean'), blend_std=('blend_f1', 'std'),
    blend_min=('blend_f1', 'min'), delta_mean=('delta_vs_exp14', 'mean'),
    delta_min=('delta_vs_exp14', 'min'), improved_seeds=('delta_vs_exp14', lambda values: int((values > 0).sum())),
    routed_rows_mean=('routed_rows', 'mean'),
)
summary['verdict'] = np.where(
    (summary['delta_mean'] > 0) & (summary['delta_min'] > 0) & (summary['routed_rows_mean'] > 0),
    'PASS', 'FAIL',
)
summary = summary.sort_values(['verdict', 'blend_mean', 'delta_min'], ascending=[False, False, False])
summary.to_csv(RESULTS_DIR / 'final_summary_3seed.csv', index=False)
display(summary)
passing = summary[summary['verdict'] == 'PASS']
print('PASS cases:', len(passing))
if len(passing):
    print('winner:', passing.iloc[0]['case'])
else:
    print('exp14 fixed_ll__hard 유지')